[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frhack/oli_ai/blob/main/notebooks/oli_ai_gradient_descent_didattica.ipynb)

# Gradient Descent — Trovare il minimo di una funzione di costo

## Cosa abbiamo già visto

1. **Regressione lineare** — dati $n$ punti $(x_i, y_i)$, cerchiamo la retta $\hat{y} = m\,x + q$ che meglio li approssima.
2. **Loss surface** (la "superficie a tazza di tè") — al variare di $(m, q)$ la perdita disegna una superficie nello spazio. Il punto migliore è il **fondo della tazza**.
3. **Formule chiuse** — per la regressione lineare sappiamo già scrivere direttamente le formule che danno $m$ e $q$ ottimali. Ma quelle formule funzionano **solo per la regressione lineare**.

## Cosa faremo oggi

Impareremo un metodo **generale** per trovare il fondo di una tazza — il **gradient descent**. Funziona per *qualsiasi* funzione di costo differenziabile, non solo per la regressione lineare. È lo stesso metodo che addestra le reti neurali e GPT.

L'idea è semplice: partiamo da un punto a caso sulla superficie, calcoliamo la **direzione di massima discesa** (il gradiente cambiato di segno), e facciamo un piccolo passo in quella direzione. Ripetiamo finché non arriviamo in fondo.

Useremo gli stessi 8 punti della demo `loss_surface_3d` vista a lezione.

## Setup

Useremo **JAX**, una libreria Python che sa **calcolare automaticamente i gradienti** delle funzioni che scriviamo. È preinstallata in Colab — non serve `pip install`.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

## 1. I dati

Gli stessi 8 punti della demo `loss_surface_3d`.

In [ ]:
x = jnp.array([0.5, 1.8, 2.5, 3.1, 4.4, 5.0, 6.2, 7.5])
y = jnp.array([3.2, 2.1, 5.8, 4.3, 7.6, 6.1, 9.4, 8.0])

plt.figure(figsize=(6, 4))
plt.scatter(x, y, s=60, color='#38bdf8', edgecolor='black', zorder=3)
plt.xlabel('x'); plt.ylabel('y'); plt.title('I nostri 8 punti')
plt.grid(alpha=0.3); plt.show()

## 2. La funzione di costo (loss)

Vogliamo una retta $\hat{y}_i = m\,x_i + q$ che passi *vicino* a tutti i punti. Misuriamo lo "sbaglio" come **somma dei quadrati degli errori**:

$$L(m, q) \;=\; \sum_{i=1}^{n} \big(y_i - (m\,x_i + q)\big)^2$$

Più $L$ è piccolo, più la retta passa vicino ai punti. Il nostro obiettivo è **trovare la coppia $(m, q)$ che rende $L$ minimo**.

Scriviamo `loss` come una funzione dei **parametri** `params = [m, q]` (un vettore di due numeri). Questa scelta vettoriale ci tornerà utile fra poco.

In [ ]:
def loss(params, x, y):
    m, q = params
    y_pred = m * x + q
    return jnp.sum((y - y_pred) ** 2)

# proviamo con una retta a caso, m=0, q=0
params = jnp.array([0.0, 0.0])
print('L(0, 0) =', float(loss(params, x, y)))

## 3. Il gradiente — la direzione di massima salita

Il **gradiente** $\nabla L$ è il vettore delle derivate parziali rispetto ai parametri:

$$\nabla L(m, q) \;=\; \left(\frac{\partial L}{\partial m},\; \frac{\partial L}{\partial q}\right)$$

Geometricamente: nel punto $(m, q)$ della superficie a tazza, il gradiente **punta nella direzione in cui $L$ cresce più rapidamente**. Quindi $-\nabla L$ punta nella direzione di **massima discesa** — esattamente dove vogliamo andare per scendere verso il fondo.

### Differenziazione automatica

Potremmo calcolare le due derivate a mano, sfruttando le regole del calcolo. Ma JAX lo fa **automaticamente** per noi: data una funzione `loss`, l'operatore `grad` produce una nuova funzione che calcola il gradiente di `loss`. Quasi come un operatore matematico: `grad(loss)` *è* il gradiente di `loss`.

Questo è enorme: vale per *qualsiasi* funzione differenziabile, non importa quanto complicata. È così che si addestrano modelli con miliardi di parametri.

In [ ]:
grad_loss = grad(loss)   # nuova funzione: ritorna il gradiente di loss

params = jnp.array([0.0, 0.0])
g = grad_loss(params, x, y)
print('gradiente in (m=0, q=0):', g)

Il gradiente è un vettore di due numeri (uno per ciascun parametro). Entrambi sono **negativi**: significa che, partendo da $(0, 0)$, la perdita *diminuisce* aumentando sia $m$ sia $q$. Coerente: la nostra retta orizzontale $y=0$ è troppo bassa, va alzata e inclinata.

## 4. Un singolo passo di gradient descent

L'algoritmo è una sola formula, da applicare ai parametri visti come **vettore**:

$$\text{params} \;\leftarrow\; \text{params} \;-\; \eta\,\nabla L(\text{params})$$

- $\eta$ (eta) è il **learning rate**: quanto è grande il passo. Lo decidiamo noi.
- Il segno **meno** ci porta nella direzione di discesa.

Proviamo un singolo passo.

In [ ]:
lr = 0.001                         # learning rate
params = jnp.array([0.0, 0.0])

print('prima:  params =', params, '  loss =', float(loss(params, x, y)))

g = grad_loss(params, x, y)
params = params - lr * g           # ← l'unico passaggio del GD

print('dopo:   params =', params, '  loss =', float(loss(params, x, y)))

La loss è scesa. Un piccolo passo nella direzione giusta.

## 5. Il loop di gradient descent

Un passo non basta: ne facciamo tanti, finché non arriviamo in fondo alla tazza. Memorizziamo il percorso (la **traiettoria**) per poterlo visualizzare.

In [ ]:
def gradient_descent(params_iniziali, lr, n_passi):
    params = params_iniziali
    traiettoria = [params]
    perdite = [float(loss(params, x, y))]
    for _ in range(n_passi):
        params = params - lr * grad_loss(params, x, y)
        traiettoria.append(params)
        perdite.append(float(loss(params, x, y)))
    return jnp.array(traiettoria), perdite

traiettoria, perdite = gradient_descent(
    params_iniziali=jnp.array([-1.0, 0.0]),
    lr=0.001,
    n_passi=500,
)

m_finale, q_finale = traiettoria[-1]
print(f'parametri finali:  m = {m_finale:.4f},  q = {q_finale:.4f}')
print(f'loss finale:       {perdite[-1]:.4f}')

## 6. Visualizziamo la discesa

Disegniamo la **superficie di loss** dall'alto (heatmap nello spazio dei parametri $(m, q)$) e sopra ci tracciamo la traiettoria del gradient descent. È esattamente la stessa scena della demo `loss_surface_3d`, ma vista da sopra.

(Il codice di questa cella è solo grafica — non è importante leggerlo.)

In [ ]:
def plotta_traiettoria(traiettorie, etichette, titolo='Discesa del gradient descent'):
    """traiettorie: lista di array (n_passi, 2). Disegna heatmap loss + traiettorie sovrapposte."""
    if not isinstance(traiettorie, list):
        traiettorie = [traiettorie]; etichette = [etichette]
    m_axis = np.linspace(-2.0, 4.0, 80)
    q_axis = np.linspace(-3.5, 6.5, 80)
    L = np.zeros((len(q_axis), len(m_axis)))
    for i, m_v in enumerate(m_axis):
        for j, q_v in enumerate(q_axis):
            L[j, i] = float(loss(jnp.array([m_v, q_v]), x, y))
    plt.figure(figsize=(8, 6))
    plt.contourf(m_axis, q_axis, L, levels=30, norm=LogNorm(), cmap='viridis')
    plt.colorbar(label='loss')
    colori = ['#f59e0b', '#ef4444', '#a78bfa', '#34d399']
    for tr, lab, c in zip(traiettorie, etichette, colori):
        tr = np.array(tr)
        plt.plot(tr[:, 0], tr[:, 1], '-o', color=c, markersize=2.5, linewidth=1.2, label=lab)
        plt.scatter(tr[0, 0], tr[0, 1], s=80, color=c, edgecolor='white', zorder=5, marker='s')
        plt.scatter(tr[-1, 0], tr[-1, 1], s=120, color=c, edgecolor='white', zorder=5, marker='*')
    plt.xlabel('m'); plt.ylabel('q'); plt.title(titolo); plt.legend(); plt.show()

plotta_traiettoria(traiettoria, etichette='lr = 0.001')

Il **quadrato** è il punto di partenza, la **stella** è dove siamo arrivati. La traiettoria scende lungo la valle della tazza fino al minimo.

Vediamo anche come decresce la loss passo per passo:

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(perdite, color='#38bdf8')
plt.xlabel('passo'); plt.ylabel('loss'); plt.yscale('log')
plt.title('La loss diminuisce a ogni passo'); plt.grid(alpha=0.3); plt.show()

## 7. L'effetto del learning rate

Il learning rate $\eta$ è il parametro più delicato del gradient descent. Vediamo cosa succede con tre valori molto diversi, partendo dallo stesso punto.

In [ ]:
params0 = jnp.array([-1.0, 0.0])

tr_basso,  _ = gradient_descent(params0, lr=0.00005, n_passi=500)
tr_giusto, _ = gradient_descent(params0, lr=0.001,   n_passi=500)
tr_alto,   _ = gradient_descent(params0, lr=0.011,   n_passi=20)

plotta_traiettoria(
    [tr_basso, tr_giusto, tr_alto],
    ['lr = 0.00005 (troppo basso)', 'lr = 0.001 (giusto)', 'lr = 0.011 (troppo alto)'],
    titolo='Tre learning rate a confronto',
)

Tre comportamenti tipici:

- **lr troppo basso**: i passi sono cortissimi, dopo 500 iterazioni siamo ancora lontani dal fondo. Funziona, ma è lento.
- **lr giusto**: scende dritto verso il minimo in poche centinaia di passi.
- **lr troppo alto**: ogni passo "salta oltre" il minimo. La traiettoria oscilla e — peggio — si allontana sempre di più. Il GD **diverge**.

Scegliere $\eta$ è una questione di *equilibrio*. Nella pratica si prova, si guarda la curva di loss, e si aggiusta.

## 8. Verifica: il GD trova davvero la soluzione giusta?

Per la regressione lineare conosciamo le **formule chiuse** che danno direttamente $(m^*, q^*)$ ottimali. Confrontiamo.

In [ ]:
# soluzione esatta tramite formule chiuse
x_medio = jnp.mean(x); y_medio = jnp.mean(y)
m_esatto = jnp.sum((x - x_medio) * (y - y_medio)) / jnp.sum((x - x_medio) ** 2)
q_esatto = y_medio - m_esatto * x_medio

# soluzione trovata dal GD (con lr giusto, più iterazioni)
tr_lungo, _ = gradient_descent(jnp.array([-1.0, 0.0]), lr=0.001, n_passi=5000)
m_gd, q_gd = tr_lungo[-1]

print(f'formula chiusa:    m = {m_esatto:.6f},  q = {q_esatto:.6f}')
print(f'gradient descent:  m = {m_gd:.6f},  q = {q_gd:.6f}')

Coincidono fino alla quarta cifra. Il GD ha trovato lo stesso minimo della formula chiusa — **senza saperla**, semplicemente seguendo i gradienti.

In [ ]:
# come appare la retta finale sui dati
xs = jnp.linspace(0, 8, 100)
plt.figure(figsize=(6, 4))
plt.scatter(x, y, s=60, color='#38bdf8', edgecolor='black', zorder=3, label='dati')
plt.plot(xs, m_gd * xs + q_gd, color='#34d399', linewidth=2, label=f'GD: y = {m_gd:.2f}x + {q_gd:.2f}')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## Riepilogo

| Concetto | Realizzazione |
|---|---|
| funzione di costo | `loss(params, x, y)` — somma dei quadrati |
| gradiente | `grad(loss)` — calcolato automaticamente da JAX |
| algoritmo | `params = params - lr * grad_loss(params, x, y)` |
| learning rate | troppo basso → lento; troppo alto → diverge |
| validazione | il GD trova la stessa soluzione delle formule chiuse |

## Perché è importante

Le formule chiuse della regressione lineare funzionano *solo* per la regressione lineare. Il **gradient descent** funziona per **qualsiasi funzione di costo differenziabile**: reti neurali, classificatori di immagini, modelli linguistici. È sempre la stessa identica idea — tre righe di codice:

```python
g = grad_loss(params, ...)
params = params - lr * g
```

Quello che cambia è solo `loss`. Tutto il resto è uguale, dai 2 parametri di oggi ai 175 miliardi di GPT-3.

## Esercizi

1. Cambia il punto di partenza in $(3, -3)$. La traiettoria è diversa? Il punto di arrivo è lo stesso?
2. Trova **a mano**, provando, il learning rate più grande con cui il GD ancora converge sui nostri dati. Cosa succede appena oltre quella soglia?
3. Modifica `loss` per usare gli **errori in valore assoluto** invece dei quadrati: $L(m,q) = \sum_i |y_i - (mx_i + q)|$. Funziona ancora? Cosa cambia? *(Suggerimento: il valore assoluto non è derivabile in zero — JAX se la cava lo stesso, ma il GD può comportarsi diversamente.)*
4. Aggiungi un nono punto **fuori dal pattern** (un *outlier*, es. $(4, 20)$) e rilancia il GD. Come cambia la retta trovata? Confronta con la versione a errori assoluti dell'esercizio 3.